In [27]:
import re
import json

# ----------------------------
# CLEAN LATEX
# ----------------------------
def clean_latex(text):
    # remove preamble
    text = re.sub(r"\\documentclass.*?\n", "", text, flags=re.S)
    text = re.sub(r"\\usepackage.*?\n", "", text)
    text = re.sub(r"\\def.*?\n", "", text)
    text = re.sub(r"\\newcommand.*?\n", "", text)

    # remove formatting noise
    text = re.sub(r"\\color\{.*?\}", "", text)
    text = re.sub(r"\\vs\{.*?\}", "", text)
    text = re.sub(r"\\mpage.*?\{", "", text)
    text = re.sub(r"\\minipage.*?\{", "", text)

    # keep structure markers but remove braces clutter
    text = text.replace("\\begin{document}", "")
    text = text.replace("\\end{document}", "")

    return text


# ----------------------------
# EXTRACT LEARNING TARGETS
# ----------------------------
def extract_learning_targets(text):
    match = re.search(r"Intended Learning Targets(.*?)\\hrulefill", text, re.S)
    if not match:
        return []

    block = match.group(1)
    items = re.findall(r"\\item\s*(.*?)(?=\\item|$)", block, re.S)
    return [i.strip() for i in items if i.strip()]


# ----------------------------
# EXTRACT EXERCISES
# ----------------------------
def extract_exercises(text):
    exercises = []

    blocks = re.split(r"\\exercise", text)

    for block in blocks[1:]:
        exercise = {}

        # raw question (until \sol or next exercise)
        q_match = re.split(r"\\sol", block)
        question = q_match[0]

        # extract subparts (a), (b), etc.
        subparts = re.findall(r"\\item\[\((.*?)\)\](.*?)(?=\\item\[\(|\\sol|$)", question, re.S)

        structured_subparts = []
        for label, content in subparts:
            structured_subparts.append({
                "label": label,
                "question": content.strip(),
                "solution": None
            })

        # extract solutions
        solutions = re.findall(r"\\sol\{(.*?)\}", block, re.S)

        # map solutions to subparts (best effort)
        for i in range(min(len(structured_subparts), len(solutions))):
            structured_subparts[i]["solution"] = solutions[i].strip()

        exercise["question"] = re.sub(r"\\sol\{.*?\}", "", question, flags=re.S).strip()
        exercise["subparts"] = structured_subparts

        # tagging (simple heuristic)
        if "rate of change" in question.lower():
            exercise["tag"] = "rates_of_change"
        elif "slope" in question.lower():
            exercise["tag"] = "linear_functions"
        else:
            exercise["tag"] = "algebra"

        # difficulty heuristic
        if len(structured_subparts) >= 4:
            exercise["difficulty"] = "hard"
        elif len(structured_subparts) >= 2:
            exercise["difficulty"] = "medium"
        else:
            exercise["difficulty"] = "easy"

        exercises.append(exercise)

    return exercises


# ----------------------------
# MAIN PIPELINE
# ----------------------------
def parse_latex_document(tex_file):
    with open(tex_file, "r", encoding="utf-8") as f:
        raw = f.read()

    cleaned = clean_latex(raw)

    output = {
        "learning_targets": extract_learning_targets(cleaned),
        "exercises": extract_exercises(cleaned)
    }

    return output


# ----------------------------
# RUN
# ----------------------------
if __name__ == "__main__":
    data = parse_latex_document("Lesson1.tex")

    with open("output.json", "w", encoding="utf-8") as f:
        json.dump(data, f, indent=2)

    print("Done → output.json")

Done → output.json


In [7]:
import json

lesson = json.load(open("/Users/joshturner/Desktop/LTI Project/Lesson1.json"))
solution = json.load(open("/Users/joshturner/Desktop/LTI Project/Lesson1_Solns.json"))

In [15]:
ex = solution["exercises"][0]

In [16]:
ex

{'exercise_number': 1,
 'raw_content': '\\exercise{\\bf\\color{ProcessBlue}F1}Respond to the following prompts.\\\\\n    \\begin{enumerate}\n    \\item[(a)] A predator-prey model describes how the sizes of two populations -- predator $P_1$ and prey $P_2$ -- are related. Suppose two points describing a predator prey model are given by $(P_1,P_2)=(100,1000)$ and $(P_1,P_2)=(300,300)$.\n        \\begin{enumerate}\n        \\item[i.] What is the net change of the prey population as the predator population changes from $100$ to $300$?\\\\\n\n            %\n            {\\Large $\\Delta P_2=$\\unline{1.5}{\\col{$300-1000=-700$}} when $P_1$ changes from\\unline{.5}{\\col{$100$}} to  \\unline{.5}{\\col{$300$}} } \n\\vs{.1}\n        \\item[ii.] What is the average rate of change of the prey population with respect to the predator population?\\\\    \n\n            {\\Large AROC $=\\frac{\\Delta P_2}{\\Delta P_1}=$\\unline{1.1}{ \\col{$\\frac{300-1000}{300-100}=-\\frac{7}{2}$}  }{.25} } %\\rule[

In [17]:
exercise_id = ex["exercise_number"]

In [18]:
exercise_id

1

In [19]:
key = f"{exercise_id}.a"

In [20]:
print(solution.get(key))

None


In [21]:
key

'1.a'

In [22]:
print(solution)

{'metadata': {'type': 'LessonCaelan', 'lesson_number': 'Lesson 1', 'title': 'Slopes \\& Rates of Change \\\\ Key', 'course': 'MATH 1190 \\& 1210', 'targets': '{\\bf\\color{ProcessBlue'}, 'exercises': [{'exercise_number': 1, 'raw_content': '\\exercise{\\bf\\color{ProcessBlue}F1}Respond to the following prompts.\\\\\n    \\begin{enumerate}\n    \\item[(a)] A predator-prey model describes how the sizes of two populations -- predator $P_1$ and prey $P_2$ -- are related. Suppose two points describing a predator prey model are given by $(P_1,P_2)=(100,1000)$ and $(P_1,P_2)=(300,300)$.\n        \\begin{enumerate}\n        \\item[i.] What is the net change of the prey population as the predator population changes from $100$ to $300$?\\\\\n\n            %\n            {\\Large $\\Delta P_2=$\\unline{1.5}{\\col{$300-1000=-700$}} when $P_1$ changes from\\unline{.5}{\\col{$100$}} to  \\unline{.5}{\\col{$300$}} } \n\\vs{.1}\n        \\item[ii.] What is the average rate of change of the prey populat

# Chunking

In [1]:
import json
file = "/Users/joshturner/Desktop/LTI Project/MathBot/JSON/lesson1.json"
with open(file, "r") as f:
    data = json.load(f)

chunks = []

#
# Learning objectives
#
for obj in data["learning_objectives"]:
    chunks.append({
        "id": f"week{data['week']}_objective_{obj['tag']}",
        "text": obj["description"],
        "metadata": {
            "week": data["week"],
            "type": "learning_objective",
            "tag": obj["tag"]
        }
    })

#
# Discussions
#
for i, discussion in enumerate(data["contents"]["other_material"]):
    chunks.append({
        "id": f"week{data['week']}_discussion_{i}",
        "text": discussion["content_plain"],
        "metadata": {
            "week": data["week"],
            "type": discussion["type"]
        }
    })

#
# Problems + subproblems
#
for problem in data["contents"]["problems"]:

    context = problem["context_plain"]

    for sub in problem["subproblems"]:

        question = sub["plain_text"]["question"]
        answer = sub["plain_text"]["answer"]

        chunk_text = f"""
Problem: {problem['name']}

Context:
{context}

Question:
{question}

Answer:
{answer}
"""

        chunks.append({
            "id": f"week{data['week']}_{problem['name']}_{sub['part']}",
            "text": chunk_text,
            "metadata": {
                "week": data["week"],
                "type": "subproblem",
                "problem_name": problem["name"],
                "part": sub["part"],
                "learning_tag": problem["learning_tag"],
                "keywords": problem["keywords"]
            }
        })

print(f"Created {len(chunks)} chunks")

Created 8 chunks


In [3]:
chunks

[{'id': 'week1_objective_1',
  'text': 'Find and distinguish the net change and average rate of change. Interpret the average rate of change graphically as the slope the respective linear function',
  'metadata': {'week': 1, 'type': 'learning_objective', 'tag': 1}},
 {'id': 'week1_discussion_0',
  'text': 'Net change is the difference between two quantities; AROC and slope are mathematically the same; Both slope and AROC are computed by the quotient of net changes',
  'metadata': {'week': 1, 'type': 'discussion'}},
 {'id': 'week1_Exercise A_a_a',
  'text': '\nProblem: Exercise A_a\n\nContext:\nA predator-prey model describes how the sizes of two populations -- predator P_1 and prey P_2 -- are related. Suppose two points describing a predator prey model are given by (P_1,P_2)=(100,1000) and (P_1,P_2)=(300,300).\n\nQuestion:\nWhat is the net change of the predator population as the prey population changes from 1000 to 300?\n\nAnswer:\nDelta P_2=300-1000=-700 when P_1 changes from 100 to 

# Sentence Transformer

In [4]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

embedding = model.encode(chunks[0]["text"])

/Users/joshturner/.pyenv/versions/calculus-rag/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13192.76it/s]


# Vector DB

In [5]:
import chromadb
from sentence_transformers import SentenceTransformer

client = chromadb.Client()

collection = client.create_collection(
    name="calculus_notes"
)

model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

for chunk in chunks:
    embedding = model.encode(chunk["text"]).tolist()

    collection.add(
        ids=[chunk["id"]],
        documents=[chunk["text"]],
        embeddings=[embedding],
        metadatas=[chunk["metadata"]]
    )

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8809.98it/s]


# Query the Vector DB

In [6]:
query = "What does the derivative represent physically?"

query_embedding = model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

print(results["documents"])

[['\nProblem: Exercise A_b\n\nContext:\nThe equation F = 9/5 C + 32 gives the relationship between temperature in degrees Fahrenheit (F) and temperature in degrees celsius (C).\n\nQuestion:\nWhat is the slope of the graph describing the relationship between F and C?\n\nAnswer:\nm = 9/5\n', 'Net change is the difference between two quantities; AROC and slope are mathematically the same; Both slope and AROC are computed by the quotient of net changes', '\nProblem: Exercise A_b\n\nContext:\nThe equation F = 9/5 C + 32 gives the relationship between temperature in degrees Fahrenheit (F) and temperature in degrees celsius (C).\n\nQuestion:\nWhat is the net change in the temperature in degrees Fahrenheit when the temperature in degrees Celsius increases by 1?\n\nAnswer:\nDelta F = 9/5 when Delta C = 1\n']]


# Response from LLM

In [ ]:
import ollama

response = ollama.chat(
    model='qwen3:8b',
    messages=[
        {
            'role': 'user',
            'content': 'What is slope?'
        }
    ]
)

print(response['message']['content'])